In [0]:
%sql
create or replace table ds_sandbox.next_uk_nextAds_predict_prod_ranked as
with t0 as (
  select *,
  case when theme_clean like '%women%' 
      and GREATEST(Womenswearconfidence_score,Menswearconfidence_score,Beautyconfidence_score,Homeconfidence_score) == Womenswearconfidence_score
      then 1
  when theme_clean like '%girls%|%boys%'
      and GREATEST(Womenswearconfidence_score,Menswearconfidence_score,Beautyconfidence_score,Homeconfidence_score,Familyconfidence_score) == Familyconfidence_score
      THEN 1
  when theme_clean like '%men%'
      and GREATEST(Womenswearconfidence_score,Menswearconfidence_score,Beautyconfidence_score,Homeconfidence_score) == Menswearconfidence_score
      THEN 1
  when theme_clean like '%beauty%'
      and GREATEST(Womenswearconfidence_score,Menswearconfidence_score,Beautyconfidence_score,Homeconfidence_score) == Beautyconfidence_score
      THEN 1
  when theme_clean like '%home%'
      and GREATEST(Womenswearconfidence_score,Menswearconfidence_score,Beautyconfidence_score,Homeconfidence_score) == Homeconfidence_score
      THEN 1
  else 0
  end as same_dept
  from ds_sandbox.next_uk_nextAds_predict_prod_complete  
  )

  , t1 as(
  select *,
  row_number() OVER (
      PARTITION BY account_number, reference_date
      ORDER BY 
      -- Primary: Recent behavior wins
      CASE WHEN views_behavior__recency = 0 THEN 999999 ELSE views_behavior__recency END ASC,
      CASE WHEN atbs_behavior__recency = 0 THEN 999999 ELSE atbs_behavior__recency END ASC,
      repurchase_ratio DESC,      
      -- Secondary: Frequency and consensus
      num_retrieval_methods DESC,
      baskets_behavior__frequency DESC,
      atbs_behavior__frequency DESC,
      views_behavior__frequency DESC,
      
      -- Tertiary: Association strength
      COALESCE(algo_baskets5__lift_top10, 0) DESC,
      COALESCE(algo_atbs5__lift_top10, 0) DESC,
      COALESCE(algo_views5__lift_top10, 0) DESC,

      -- dept segment
      same_dept desc,

      theme_clean  -- Deterministic tiebreaker
  ) AS simple_rules_rank
  from t0
  )
  , final as (
  select *,
  case
  when views_behavior__recency >0 and views_behavior__recency <9999 then 'views_recency'
  when atbs_behavior__recency >0 and atbs_behavior__recency <9999 then 'atbs_recency'
  when repurchase_ratio >0 then 'repurchase_ratio'
  when num_retrieval_methods >0 and baskets_behavior__frequency >0 then 'ret_met_baskets_frequency'
  when num_retrieval_methods >0 and atbs_behavior__frequency >0 then 'ret_met_atbs_frequency'
  when num_retrieval_methods >0 and views_behavior__frequency >0 then 'ret_met_views_frequency'
  when num_retrieval_methods >0 and algo_baskets5__lift_top10 >0 then 'ret_met_algo_baskets5_lift_top10'
  when num_retrieval_methods >0 and algo_atbs5__lift_top10 >0 then 'ret_met_algo_atbs5_lift_top10'
  when num_retrieval_methods >0 and algo_views5__lift_top10 >0 then 'ret_met_algo_views5_lift_top10'
  when num_retrieval_methods >0 and same_dept = 1 then 'ret_met_same_dept'
  when baskets_behavior__frequency >0 then 'baskets_frequency'
  when atbs_behavior__frequency >0 then 'atbs_frequency'
  when views_behavior__frequency >0 then 'views_frequency'
  when algo_baskets5__lift_top10 >0 then 'algo_baskets5_lift_top10'
  when algo_atbs5__lift_top10 >0 then 'algo_atbs5_lift_top10'
  when algo_views5__lift_top10 >0 then 'algo_views5_lift_top10'
  when same_dept = 1 then 'same_dept'
  else 'theme'
  end as rules_rank_source
  from t1
  -- where account_number = 'M52K2576'
  GROUP BY all
  ORDER BY simple_rules_rank
  )

  select distinct 
  *
  from final
  -- where simple_rules_rank <= 100
  order by account_number, simple_rules_rank